# Strands Agents on AWS Bedrock — Classroom Ready Notebook

This notebook introduces simple agent building using the **Strands Agents SDK** in a SageMaker notebook with **Amazon Bedrock**.

## Learning goals
By the end of this notebook, learners should be able to:
- understand what an agent is
- create a simple Strands agent with a system prompt
- create a Strands agent that can use Python tools
- observe the difference between a prompt-only agent and a tool-using agent
- build two small agents as classroom activities


## Step 0 — Install packages

Run this cell only if the packages are not already available in the SageMaker notebook kernel.

In [7]:
# Uncomment if needed
#!pip install -q strands-agents strands-agents-tools boto3 pandas

## Step 1 — Imports

This step imports the packages used in the notebook.

In [8]:
import boto3
import pandas as pd
from datetime import datetime

from strands import Agent, tool
from strands.models import BedrockModel

## Step 2 — Bedrock configuration


In [9]:
AWS_REGION = boto3.session.Session().region_name or "us-east-1"

# Change this if your classroom uses a different Bedrock model.
'''
Avaialble Models:

amazon.nova-lite-v1:0 
amazon.nova-micro-v1:0 
amazon.nova-pro-v1:0

'''
MODEL_ID = "amazon.nova-lite-v1:0"

# MODEL_ID = "amazon.nova-micro-v1:0:24k"

bedrock_model = BedrockModel(
    model_id=MODEL_ID,
    region_name=AWS_REGION,
    temperature=0.1,
)

print("Region:", AWS_REGION)
print("Model ID:", MODEL_ID)

Region: us-east-1
Model ID: amazon.nova-lite-v1:0


## Step 3 — Build a simple explanation agent

A simple agent usually has three building blocks:
- **LLM** (the reasoning engine)
- **Tools** (optional actions the agent can take)
- **Memory** (optional context carried across turns)

We start with the simplest version: an agent with only an LLM and a system prompt.

### What learners should notice
- the agent behavior changes because of the system prompt
- no tools are used here
- the agent can explain concepts, but it cannot take actions

In [10]:
concept_agent = Agent(
    model=bedrock_model,
    system_prompt="""
You are a teaching assistant for a first class on AI agents.
Your job is to explain concepts simply.
Use:
1. a one-line definition
2. a short example
3. one caution
Keep the answer short.
""",
)

concept_question = "What is the difference between a chatbot and an agent?"
concept_response = concept_agent(concept_question)
print(concept_response)

1. **Definition**: A chatbot follows predefined rules to respond to inputs, while an agent can make decisions and take actions autonomously.
2. **Example**: A chatbot answers FAQs on a website; an agent books flights based on user preferences and availability.
3. **Caution**: Don't expect a chatbot to understand complex queries or adapt to new situations without human intervention.1. **Definition**: A chatbot follows predefined rules to respond to inputs, while an agent can make decisions and take actions autonomously.
2. **Example**: A chatbot answers FAQs on a website; an agent books flights based on user preferences and availability.
3. **Caution**: Don't expect a chatbot to understand complex queries or adapt to new situations without human intervention.



## Step 4 — Add tools to the agent

Now we create a few small Python tools.

These tools are intentionally simple:
- one tool returns the current classroom time
- one tool calculates a percentage
- one tool formats a short learner progress message

The important idea is:

**the agent decides when to call a tool, and Python executes it**

In [11]:
@tool
def get_current_class_time() -> str:
    """Return the current date and time for the classroom session."""
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

@tool
def calculate_percentage(score: float, total: float) -> float:
    """Calculate percentage from score and total."""
    if total == 0:
        raise ValueError("total must not be zero")
    return round((score / total) * 100, 2)

@tool
def learner_progress_message(name: str, score: float) -> str:
    """Create a short progress message for a learner."""
    if score >= 85:
        level = "excellent"
    elif score >= 70:
        level = "good"
    elif score >= 50:
        level = "developing"
    else:
        level = "needs support"
    return f"{name} is currently at {level} performance."

## Step 5 — Agent with tools

This agent has access to the tools created above.

### What learners should notice
- the user asks for more than one thing
- the agent can decide which tools to use
- the final answer is usually more practical because tools do part of the work

In [18]:
classroom_agent = Agent(
    model=bedrock_model,
    tools=[get_current_class_time, calculate_percentage, learner_progress_message],
    system_prompt="""
You are a classroom operations agent.
Use tools whenever they help you answer correctly.
Be concise and practical.
"""
)



In [19]:
# The agent will call @get_current_class_time tool
tool_response1 = classroom_agent(
    """
    Please do these tasks:
    Tell me the current class time.
    """
)

print(tool_response1)

<thinking>The User has asked for the current class time. I can use the 'get_current_class_time' tool to retrieve this information.</thinking>

Tool #1: get_current_class_time
<thinking>I have retrieved the current class time using the 'get_current_class_time' tool. I can now provide this information to the User.</thinking>

The current class time is 2026-04-20 19:30:03.<thinking>I have retrieved the current class time using the 'get_current_class_time' tool. I can now provide this information to the User.</thinking>

The current class time is 2026-04-20 19:30:03.



In [20]:
# The agent will call @calculate_percentage tool
tool_response2 = classroom_agent(
    """
Please do these tasks:
Calculate the percentage for 42 out of 50.
"""
)

print(tool_response2)

<thinking>The User has asked to calculate the percentage for a score of 42 out of 50. I can use the 'calculate_percentage' tool to perform this calculation.</thinking> 
Tool #2: calculate_percentage
<thinking>I have calculated the percentage using the 'calculate_percentage' tool. I can now provide this information to the User.</thinking>

The percentage for a score of 42 out of 50 is 84.0%.<thinking>I have calculated the percentage using the 'calculate_percentage' tool. I can now provide this information to the User.</thinking>

The percentage for a score of 42 out of 50 is 84.0%.



In [21]:
# The agent will call @learner_progress_message tool
tool_response3 = classroom_agent(
    """
Please do these tasks:
Create a short progress message for Riya using score 84.
"""
)

print(tool_response3)

<thinking>The User has asked to create a short progress message for Riya with a score of 84. I can use the 'learner_progress_message' tool to generate this message.</thinking> 
Tool #3: learner_progress_message
<thinking>I have created a progress message for Riya using the 'learner_progress_message' tool. I can now provide this message to the User.</thinking>

Here is the progress message for Riya: "Riya is currently at good performance."<thinking>I have created a progress message for Riya using the 'learner_progress_message' tool. I can now provide this message to the User.</thinking>

Here is the progress message for Riya: "Riya is currently at good performance."



## Step 6 — Compare the two agent styles

This small comparison table helps learners see the difference between:
- an agent without tools
- an agent with tools

### Example comparison

| Example | Agent type | Uses tools? | Best for | Sample task |
|---|---|---:|---|---|
| Example 1 | Prompt-shaped agent | No | Explaining concepts clearly | “What is the difference between a chatbot and an agent?” |
| Example 2 | Tool-using agent | Yes | Doing small actions or calculations | “Tell me the current class time, calculate 42 out of 50, and create a progress message for Riya.” |

### What changes between the two?

- **Agent without tools**
  - Can explain, summarize, rewrite, and reason from the model’s knowledge
  - Best when the task is mainly language-based
  - Cannot reliably perform external actions unless tools are provided

- **Agent with tools**
  - Can still explain concepts
  - Can also call Python functions when needed
  - Best when the task needs calculation, formatting, lookup, or action execution

### Simple intuition

- A **prompt-only agent** is like a smart assistant that can think and talk.
- A **tool-using agent** is like a smart assistant that can think, talk, and also use instruments when required.

## Step 7 — Activity 1: Build a small business helper agent

### Goal
Build an agent that helps with simple store operations.

### What this agent should do
- calculate bill totals
- apply discount percentages
- produce a short final message for the customer


In [16]:
# Your code goes here

## Step 8 - Activity 2:  Domain related Agent Building

Build a “Smart Payroll Assistant Agent” for a mid-sized company. The HR team wants to automate repetitive payroll tasks while minimizing errors.

The agent should take employee data as input and intelligently use tools (functions) to:

Core Responsibilities
Calculate gross salary
Apply tax deductions and statutory components (e.g., PF, professional tax)
Handle variable components (bonuses, overtime, unpaid leaves)
Compute net salary
Generate a human-readable payslip summary

In [17]:
# Your code goes here